<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       HMM Event Sequence Classification — Teradata
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style = 'font-size:16px;font-family:Arial'><b>Steps:</b></p>
<ol style = 'font-size:16px;font-family:Arial'>
    <li>Connect to TeradataCloud</li>
    <li>Create HMM tables</li>
    <li>Install 8 stored procedures</li>
    <li>Binary HMM (ApplyMortgage vs NoApplication)  3/4/5 states</li>
    <li>Multiclass HMM (all Apply\* products)  3 states</li>
    <li>Full metrics and visualization</li>
    <li>Model interpretation (emissions, transitions)</li>
    <b>All SQL, SP definitions, training/scoring workflows, and plotting live in **`hmm_support.py`**.</b>
</ol>

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>1. Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial'>Connect to TeradataCloud using <code>create_context</code> from the teradataml Python library. </p>

In [ ]:
from getpass import getpass
from teradataml import *
import os

# Suppress warnings
import warnings

warnings.filterwarnings('ignore')
display.suppress_vantage_runtime_warnings = True

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env")
    # Create the Context
    eng = create_context(host=env_vars.get("host"), username=env_vars.get("username"), password=env_vars.get("my_variable"))
    execute_sql('''SET query_band='DEMO=3._Bank_Click_Stream_-_Outcome_Prediction_with_Hidden_Markov_Model.ipynb;' UPDATE FOR SESSION;''')
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

## 1. Setup & Connect

In [ ]:
from hmm_support import *

In [ ]:
ctx = connect_vantage(env_vars.get("host"), env_vars.get("username"), env_vars.get("my_variable"), "TD2")

In [ ]:
verify_source_tables("DEMO_Bank.Session_Events_Train", "Demo_Bank.Session_Events_Test")

## 2. Create Tables & Install Stored Procedures

In [ ]:
create_hmm_tables(drop_existing=True)

In [ ]:
install_all_stored_procedures()

## 3. Binary Classification: ApplyMortgage vs NoApplication

Train a positive HMM on sessions that end in `ApplyMortgage` and a negative
HMM on sessions with no Apply event. Score the test set with both, compute
log-likelihood ratio → sigmoid → probability. Repeat for 3, 4, and 5 states.

In [ ]:
df = DataFrame.from_query("SEL * FROM DEMO_Bank.Session_Events_Train")
df

In [ ]:
TARGET = 'ApplyMortgage'
TRAIN_TABLE = 'DEMO_Bank.Session_Events_Train'
TEST_TABLE = 'DEMO_Bank.Session_Events_Test'

binary_results = {}

# for n_states in [3, 4, 5]:
# for experiment run n_states 3 is taken, with enough compute resource n_states 3,4,5 can be taken
for n_states in [3]:
    
    print(f"\n{'='*60}")
    print(f"  {n_states} HIDDEN STATES")
    print(f"{'='*60}")

    # Train
    pos_id, neg_id = train_binary_hmm(TARGET, TRAIN_TABLE, n_states=n_states)
    pos_id = f'bin_{TARGET}_pos_{n_states}s'
    neg_id = f'bin_{TARGET}_neg_{n_states}s'
    # Score
    scores = score_binary_hmm(pos_id, neg_id, TEST_TABLE)

    # Evaluate
    metrics = evaluate_binary(scores, TEST_TABLE, TARGET)

    binary_results[n_states] = metrics

### Binary Results Summary

In [ ]:
rows = []
for ns, r in binary_results.items():
    m05 = r['thresholds'][0.5]
    rows.append({
        'States': ns, 'AUC': r['auc'],
        'Accuracy': m05['accuracy'], 'Precision': m05['precision'],
        'Recall': m05['recall'], 'F1': m05['f1']
    })
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
plot_binary_results(binary_results, TARGET)

## 4. Multiclass Classification: All Apply\* Products

Train one HMM per Apply\* class plus NoApplication. Classify each test session
by assigning it to whichever class HMM gives the highest log-likelihood.

In [ ]:
mc_model_ids = train_multiclass_hmm(TRAIN_TABLE, n_states=3)

In [ ]:
mc_scores, mc_class_names = score_multiclass_hmm(mc_model_ids, TEST_TABLE)

In [ ]:
mc_results = evaluate_multiclass(mc_scores, TEST_TABLE)

In [ ]:
plot_multiclass_results(mc_results)

## 5. Training Convergence

In [ ]:
# Combine all model_ids for convergence plot
all_model_ids = {}
for cls, mid in mc_model_ids.items():
    all_model_ids[cls] = mid
# Add binary models from best state count
best_ns = max(binary_results, key=lambda k: binary_results[k]['auc'])
all_model_ids[f'{TARGET}_pos'] = f'bin_{TARGET}_pos_{best_ns}s'
all_model_ids[f'{TARGET}_neg'] = f'bin_{TARGET}_neg_{best_ns}s'

plot_training_convergence(all_model_ids)

## 6. Combined Results Plot

In [ ]:
plot_all_results(binary_results, mc_results, TARGET)

## 7. Model Interpretation

What do the hidden states *mean*? Examine the top emission events per state
and the transition dynamics to understand customer journey phases.

In [ ]:
for cls in ['ApplyMortgage', 'ApplyCreditCard', 'NoApplication']:
    mid = mc_model_ids.get(cls)
    if mid:
        print(f"\n{'='*50}")
        print(f"  {cls}")
        print(f"{'='*50}")
        print_model_interpretation(mid)

## 8. Cleanup (Optional)

Uncomment to remove all HMM artifacts from Vantage.

In [ ]:
drop_hmm_tables()
drop_all_stored_procedures()
disconnect()